# `generate_code` — Exhaustive Path Demo

This notebook exercises **every code path** in `generate_code_from_plan` by
constructing `DataProfile` + `ForecastPlan` pairs and printing the generated code.

The generated code is **not executed** — only printed and syntax-checked via `compile()`.

In [1]:
import sys
sys.path.insert(0, "..")

from skforecast_ai.schemas import DataProfile, ForecastPlan, PreprocessingStep
from skforecast_ai import ForecastingAssistant

assistant = ForecastingAssistant()


def show(plan, profile):
    """Generate code, syntax-check it, and print."""
    code = assistant.generate_code_from_plan(plan=plan, data_profile=profile)
    compile(code, "<demo>", "exec")  # raises SyntaxError if invalid
    print(code)
    print("\n" + "=" * 70 + "\n")

## 1. Single Series

### 1.1 Recursive minimal (LGBMRegressor, no exog, no intervals)

In [2]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterRecursive",
    estimator         = "LGBMRegressor",
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3, 4, 5, 6, 7], "dropna_from_series": False},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Minimal recursive.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
    lags               = [1, 2, 3, 4, 5, 6, 7],
    dropna_from_series = False,
)

# Fit
forecaster.fit(y=data_train['y'])

# Predi

### 1.2 Recursive full (Ridge, exog+categorical, window_features, transformers, differentiation)

In [3]:
profile = DataProfile(
    n_series         = 1,
    n_observations   = 720,
    target           = "sales",
    index_type       = "datetime",
    frequency        = "h",
    exog_columns     = ["temperature", "day_of_week"],
    categorical_exog = ["day_of_week"],
    end_train        = "2024-01-24",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterRecursive",
    estimator         = "Ridge",
    steps             = 24,
    frequency         = "h",
    forecaster_kwargs = {
        "lags": [1, 7, 24],
        "window_features": [
            {"stats": ["mean", "std"], "window_sizes": 24},
            {"stats": ["mean"], "window_sizes": 168},
        ],
        "transformer_y": "StandardScaler",
        "transformer_exog": "StandardScaler",
        "categorical_features": "auto",
        "differentiation": 1,
        "dropna_from_series": True,
    },
    interval          = [5, 95],
    interval_method   = "bootstrapping",
    use_exog          = True,
    explanation       = "Full config.",
)

show(plan, profile)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('h')
data = data.sort_index()

# Train/test split
end_train = '2024-01-24'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]
exog_features = ['temperature', 'day_of_week']

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

window_features = RollingFeatures(
    stats      

### 1.3 Recursive exog numeric-only (plain StandardScaler transformer_exog)

In [4]:
profile = DataProfile(
    n_series         = 1,
    n_observations   = 720,
    target           = "sales",
    index_type       = "datetime",
    frequency        = "h",
    exog_columns     = ["temperature", "humidity"],
    categorical_exog = [],
    end_train        = "2024-01-24",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterRecursive",
    estimator         = "LGBMRegressor",
    steps             = 24,
    frequency         = "h",
    forecaster_kwargs = {
        "lags": [1, 2, 3, 24],
        "transformer_exog": "StandardScaler",
        "dropna_from_series": False,
    },
    interval_method   = "bootstrapping",
    use_exog          = True,
    explanation       = "Exog numeric only.",
)

show(plan, profile)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('h')
data = data.sort_index()

# Train/test split
end_train = '2024-01-24'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]
exog_features = ['temperature', 'humidity']

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

transformer_exog = StandardScaler()

# Create forecaster
forecaster = ForecasterRecursive(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
 

### 1.4 ForecasterDirect (steps in constructor)

In [5]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterDirect",
    estimator         = "Ridge",
    steps             = 14,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3, 4, 5, 6, 7], "steps": 14, "dropna_from_series": False},
    interval_method   = "bootstrapping",
    use_exog          = False,
    explanation       = "Direct forecaster.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.direct import ForecasterDirect

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster
forecaster = ForecasterDirect(
    estimator          = Ridge(),
    steps              = 14,
    lags               = [1, 2, 3, 4, 5, 6, 7],
    dropna_from_series = False,
)

# Fit
forecaster.fit(
    y                         = data_t

### 1.5 Unknown estimator (TODO import)

In [6]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "single_series",
    forecaster        = "ForecasterRecursive",
    estimator         = "GradientBoostingRegressor",
    steps             = 10,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3], "dropna_from_series": False},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Unknown estimator.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from __main__ import GradientBoostingRegressor  # TODO: replace with correct import for GradientBoostingRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator          = GradientBoostingRegressor(),
    lags               = [1, 2, 3],
    dropna_from_series = False,


### 1.6 Preprocessing steps (drop_duplicates)

In [7]:
profile = DataProfile(
    n_series               = 1,
    n_observations         = 365,
    target                 = "y",
    index_type             = "datetime",
    frequency              = "D",
    has_duplicate_timestamps = True,
    end_train              = "2024-10-18",
)

plan = ForecastPlan(
    task_type           = "single_series",
    forecaster          = "ForecasterRecursive",
    estimator           = "LGBMRegressor",
    steps               = 10,
    frequency           = "D",
    forecaster_kwargs   = {"lags": [1, 2, 3, 7], "dropna_from_series": False},
    interval_method     = None,
    use_exog            = False,
    preprocessing_steps = [
        PreprocessingStep(
            action="drop_duplicates",
            reason="Duplicate timestamps cause errors.",
            code_snippet="data = data[~data.index.duplicated(keep='first')]",
            blocking=True,
        ),
    ],
    explanation = "With preprocessing.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.sort_index()

# Preprocessing
data = data[~data.index.duplicated(keep='first')]
data = data.asfreq('D')

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
    lags               = [1, 2, 3, 7],
    dropna_from_series = 

## 2. Multi-Series

### 2.1 Long format (pivot_table, conformal intervals)

In [8]:
profile = DataProfile(
    data_format      = "long",
    n_series         = 3,
    n_observations   = 300,
    target           = "sales",
    date_column      = "date",
    series_id_column = "series_id",
    index_type       = "datetime",
    frequency        = "D",
    end_train        = "2024-08-05",
)

plan = ForecastPlan(
    task_type         = "multi_series",
    forecaster        = "ForecasterRecursiveMultiSeries",
    estimator         = "LGBMRegressor",
    steps             = 14,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3, 4, 5, 6, 7], "encoding": "ordinal", "dropna_from_series": False},
    interval_method   = "conformal",
    use_exog          = False,
    explanation       = "Long format multi-series.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import reshape_series_long_to_dict
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Load data
data = pd.read_csv('data.csv')
data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')

# Reshape to dict format (optimal for ForecasterRecursiveMultiSeries)
series_dict = reshape_series_long_to_dict(
    data      = data,
    series_id = 'series_id',
    index     = 'date',
    values    = 'sales',
    freq      = 'D',
)

# Train/test split
end_train = '2024-08-05'  # 80% of data, adjust to change the split point
series_dict_train = {k: v.loc[:end_train] for k, v in series_dict.items()}
series_dict_test  = {k: v.loc[v.index > end_train] for k, v in series_dict.items()}

# Create forecaster
forecaster = ForecasterRecursiveMultiSeries(
    estimator 

### 2.2 Wide format (no pivot, no intervals)

In [9]:
profile = DataProfile(
    data_format    = "wide",
    n_series       = 3,
    n_observations = 300,
    target         = ["series_a", "series_b", "series_c"],
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-08-05",
)

plan = ForecastPlan(
    task_type         = "multi_series",
    forecaster        = "ForecasterRecursiveMultiSeries",
    estimator         = "LGBMRegressor",
    steps             = 14,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3, 7], "encoding": "ordinal", "dropna_from_series": False},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Wide format multi-series.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

# Reshape to dict format (optimal for ForecasterRecursiveMultiSeries)
series_dict = data[['series_a', 'series_b', 'series_c']].to_dict('series')

# Train/test split
end_train = '2024-08-05'  # 80% of data, adjust to change the split point
series_dict_train = {k: v.loc[:end_train] for k, v in series_dict.items()}
series_dict_test  = {k: v.loc[v.index > end_train] for k, v in series_dict.items()}

# Create forecaster
forecaster = ForecasterRecursiveMultiSeries(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
    lags               = [1, 2, 3, 7],
    encoding           = 'ordinal',
    dropna_from_serie

### 2.3 Long format with exog+categorical (transformer_series, transformer_exog, window_features)

In [10]:
profile = DataProfile(
    data_format      = "long",
    n_series         = 3,
    n_observations   = 300,
    target           = "sales",
    date_column      = "date",
    series_id_column = "store_id",
    index_type       = "datetime",
    frequency        = "D",
    exog_columns     = ["temperature", "holiday"],
    categorical_exog = ["holiday"],
    end_train        = "2024-08-05",
)

plan = ForecastPlan(
    task_type         = "multi_series",
    forecaster        = "ForecasterRecursiveMultiSeries",
    estimator         = "LGBMRegressor",
    steps             = 7,
    frequency         = "D",
    forecaster_kwargs = {
        "lags": [1, 2, 3, 7],
        "encoding": "ordinal",
        "window_features": [{"stats": ["mean", "std"], "window_sizes": 7}],
        "transformer_series": "StandardScaler",
        "transformer_exog": "StandardScaler",
        "categorical_features": "auto",
        "differentiation": 1,
        "dropna_from_series": False,
    },
    interval_method   = None,
    use_exog          = True,
    explanation       = "Multi-series full config.",
)

show(plan, profile)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import RollingFeatures, reshape_series_long_to_dict, reshape_exog_long_to_dict
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Load data
data = pd.read_csv('data.csv')
data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')

# Reshape to dict format (optimal for ForecasterRecursiveMultiSeries)
series_dict = reshape_series_long_to_dict(
    data      = data,
    series_id = 'store_id',
    index     = 'date',
    values    = 'sales',
    freq      = 'D',
)

exog_dict = reshape_exog_long_to_dict(
    data      = data[['store_id', 'date', 'temperature', 'holiday']],
    series_id = 'store_id',
    index     = 'date',
    freq      = 'D',
)

# T

## 3. Multivariate

### 3.1 Wide, no exog

In [11]:
profile = DataProfile(
    data_format    = "wide",
    n_series       = 3,
    n_observations = 365,
    target         = ["series_a", "series_b", "series_c"],
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "multivariate",
    forecaster        = "ForecasterDirectMultiVariate",
    estimator         = "Ridge",
    steps             = 10,
    frequency         = "D",
    forecaster_kwargs = {"lags": [1, 2, 3], "steps": 10, "dropna_from_series": False},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Multivariate basic.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.direct import ForecasterDirectMultiVariate

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

# Create forecaster
forecaster = ForecasterDirectMultiVariate(
    estimator          = Ridge(),
    level              = 'series_a',
    steps              = 10,
    lags               = [1, 2, 3],
    dropna_from_series = False,
)

# Fit
forecaster.fit(series=data_train)

# Predict
steps = 10
predictions = forecaster.predict(steps=steps)
print(predictions)

# Evaluate on test set
actual = data_test['series_a'].iloc[:steps]
mae  = mean_absolute_error(a

### 3.2 Wide, with exog, intervals, differentiation

In [12]:
profile = DataProfile(
    data_format    = "wide",
    n_series       = 3,
    n_observations = 365,
    target         = ["series_a", "series_b", "series_c"],
    index_type     = "datetime",
    frequency      = "D",
    exog_columns   = ["temperature", "promo"],
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "multivariate",
    forecaster        = "ForecasterDirectMultiVariate",
    estimator         = "LGBMRegressor",
    steps             = 10,
    frequency         = "D",
    forecaster_kwargs = {
        "lags": [1, 2, 3],
        "steps": 10,
        "differentiation": 1,
        "dropna_from_series": False,
    },
    interval_method   = "bootstrapping",
    use_exog          = True,
    explanation       = "Multivariate with exog and intervals.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.direct import ForecasterDirectMultiVariate

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
series_cols = ['series_a', 'series_b', 'series_c']
exog_features = ['temperature', 'promo']
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

# Create forecaster
forecaster = ForecasterDirectMultiVariate(
    estimator          = LGBMRegressor(random_state=123, verbose=-1),
    level              = 'series_a',
    steps              = 10,
    lags               = [1, 2, 3],
    differentiation    = 1,
    dropna_from_series = False,
)

# Fit
forecaster.fit(
    series                    = data_train[series

## 4. Statistical

### 4.1 Arima (auto, daily → m=7, no exog)

In [13]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "statistical",
    forecaster        = "ForecasterStats",
    estimator         = None,
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {"stats_model": "Arima"},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Auto-ARIMA daily.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.stats import Arima
from skforecast.recursive import ForecasterStats

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster (Auto-ARIMA)
forecaster = ForecasterStats(
    estimator = Arima(order=None, seasonal_order=None, m=7),
)

# Fit
forecaster.fit(y=data_train['y'])

# Predict
steps = 30
predictions = forecaster.predict(steps=steps)
print(predictions)

# Ev

In [14]:
profile = DataProfile(
    n_series         = 1,
    n_observations   = 720,
    target           = "sales",
    index_type       = "datetime",
    frequency        = "h",
    exog_columns     = ["temperature", "day_of_week"],
    categorical_exog = ["day_of_week"],
    end_train        = "2024-01-24",
)

plan = ForecastPlan(
    task_type         = "statistical",
    forecaster        = "ForecasterStats",
    estimator         = None,
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {"stats_model": "Arima"},
    interval          = [5, 95],
    interval_method   = "native",
    use_exog          = True,
    explanation       = "Auto-ARIMA daily.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.stats import Arima
from skforecast.recursive import ForecasterStats

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('h')
data = data.sort_index()

# Train/test split
end_train = '2024-01-24'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]
exog_features = ['temperature', 'day_of_week']

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster (Auto-ARIMA)
forecaster = ForecasterStats(
    estimator = Arima(order=None, seasonal_order=None, m=24),
)

# Fit
forecaster.fit(y=data_train['sales'], exog=data_train[exog_features])

# Predi

## 5. Foundation

### 5.1 Chronos minimal (single series, no exog, no intervals)

In [15]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "foundation",
    forecaster        = "ForecasterFoundation",
    estimator         = None,
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Chronos minimal.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.foundation import FoundationModel, ForecasterFoundation

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

series = data['y']

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]

# Create foundation model (chronos-2-small)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-small',
    context_length = 2048,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=series_train)

# Predict
steps = 30
predictions = forecaster.predict(steps=steps)
print(predictions)

# Evaluate on test set
actual = serie

### 5.2 Chronos custom (model_id, context_length, exog, custom quantiles)

In [16]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 500,
    target         = "y",
    index_type     = "datetime",
    frequency      = "h",
    exog_columns   = ["temperature"],
    end_train      = "2024-01-17",
)

plan = ForecastPlan(
    task_type         = "foundation",
    forecaster        = "ForecasterFoundation",
    estimator         = None,
    steps             = 24,
    frequency         = "h",
    forecaster_kwargs = {"model_id": "autogluon/chronos-2-base", "context_length": 4096},
    interval          = [20, 80],
    interval_method   = "native",
    use_exog          = True,
    explanation       = "Custom Chronos with exog.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.foundation import FoundationModel, ForecasterFoundation

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('h')
data = data.sort_index()

series = data['y']
exog = data[['temperature']]

# Train/test split
end_train = '2024-01-17'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]
exog_train = exog.loc[:end_train]
exog_test  = exog.loc[exog.index > end_train]

# Create foundation model (chronos-2-small)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-small',
    context_length = 2048,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=series_train, exog=exog_train)

### 5.3 Foundation multi-series (wide format, levels in predict)

In [17]:
profile = DataProfile(
    data_format    = "wide",
    n_series       = 3,
    n_observations = 365,
    target         = ["series_a", "series_b", "series_c"],
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "foundation",
    forecaster        = "ForecasterFoundation",
    estimator         = None,
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {},
    interval_method   = None,
    use_exog          = False,
    explanation       = "Foundation multi-series.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.foundation import FoundationModel, ForecasterFoundation

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

series = data[['series_a', 'series_b', 'series_c']]

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]

# Create foundation model (chronos-2-small)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-small',
    context_length = 2048,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=series_train)

# Predict
steps = 30
predictions = forecaster.predict(steps=steps, levels=['series_a', 'ser

### 5.4 Foundation with default quantiles (no custom interval)

In [18]:
profile = DataProfile(
    n_series       = 1,
    n_observations = 365,
    target         = "y",
    index_type     = "datetime",
    frequency      = "D",
    end_train      = "2024-10-18",
)

plan = ForecastPlan(
    task_type         = "foundation",
    forecaster        = "ForecasterFoundation",
    estimator         = None,
    steps             = 30,
    frequency         = "D",
    forecaster_kwargs = {},
    interval          = [10, 90],
    interval_method   = "native",
    use_exog          = False,
    explanation       = "Foundation default quantiles.",
)

show(plan, profile)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.foundation import FoundationModel, ForecasterFoundation

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

series = data['y']

# Train/test split
end_train = '2024-10-18'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]

# Create foundation model (chronos-2-small)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-small',
    context_length = 2048,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=series_train)

# Predict quantiles (native)
steps = 30
predictions = forecaster.predict_quantiles(
    steps     = steps,
    quantiles = [

## 6. End-to-End with Real Data

In a real situation, you never construct `PreprocessingStep` objects manually.
The assistant pipeline does it for you:

1. **`assistant.profile(data, target)`** — inspects the DataFrame, detects issues
   (duplicates, missing values, categorical columns, etc.) and recommends forecaster + estimator.
2. **`assistant.generate_plan(profile, steps)`** — derives lags, intervals,
   NaN handling, and **preprocessing steps** automatically from the detected issues.
3. **`assistant.generate_code_from_plan(plan, profile)`** — emits the script.

The `preprocessing_steps` field in the `ForecastPlan` only contains
**non-standard remediation steps** (e.g. drop duplicates, encode target).
Standard operations (sort, set index, asfreq) are always emitted by the
data-loading block.

### 6.1 Single series — clean data (no preprocessing steps generated)

In [19]:
import numpy as np
import pandas as pd

# Create a clean single-series dataset (no issues)
data_clean = pd.DataFrame(
    {"y": np.sin(np.linspace(0, 8 * np.pi, 365)) + np.random.default_rng(42).normal(0, 0.3, 365)},
    index=pd.date_range("2023-01-01", periods=365, freq="D"),
)

# Full pipeline
forecaster_profile = assistant.profile(data=data_clean, target="y")

print("Recommended forecaster:", forecaster_profile.forecaster)
print("Recommended estimator:", forecaster_profile.estimator)
print()

plan = assistant.generate_plan(forecaster_profile, steps=30)

print("Preprocessing steps:", plan.preprocessing_steps)
print()

code = assistant.generate_code_from_plan(plan=plan, data_profile=forecaster_profile.data_profile)
compile(code, "<demo>", "exec")
print(code)

Recommended forecaster: ForecasterRecursive
Recommended estimator: LGBMRegressor

Preprocessing steps: []

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2023-10-19'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean'],
    window_sizes

### 6.2 Single series — data with duplicates (drop_duplicates step generated)

In [20]:
# Create a dataset with duplicate timestamps
rng = np.random.default_rng(42)
dates = pd.date_range("2023-01-01", periods=365, freq="D")
data_dupes = pd.DataFrame(
    {"y": rng.normal(100, 10, 365)},
    index=dates,
)
# Inject 5 duplicate timestamps
data_dupes = pd.concat([data_dupes, data_dupes.iloc[:5]])
data_dupes = data_dupes.sort_index()

print(f"Has duplicates: {data_dupes.index.duplicated().any()}")
print(f"Shape: {data_dupes.shape}")
print()

# Full pipeline — the assistant detects duplicates automatically
forecaster_profile = assistant.profile(data=data_dupes, target="y")
plan = assistant.generate_plan(forecaster_profile, steps=14)

print("Preprocessing steps:")
for step in plan.preprocessing_steps:
    print(f"  - {step.action}: {step.reason}")
    print(f"    code: {step.code_snippet}")
print()

code = assistant.generate_code_from_plan(plan=plan, data_profile=forecaster_profile.data_profile)
compile(code, "<demo>", "exec")
print(code)

Has duplicates: True
Shape: (370, 1)

Preprocessing steps:
  - drop_duplicates: Duplicate timestamps cause errors in skforecast.
    code: data = data[~data.index.duplicated(keep='first')]

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.sort_index()

# Preprocessing
data = data[~data.index.duplicated(keep='first')]
data = data.asfreq('D')

# Train/test split
end_train = '2023-10-18'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()}

### 6.3 Multi-series long format (no false duplicate detection)

In [21]:
# Create a long-format multi-series dataset
# Same dates appear for each series — this is NOT duplicates
rng = np.random.default_rng(123)
dates = pd.date_range("2023-01-01", periods=200, freq="D")
series_ids = ["store_A", "store_B", "store_C"]

rows = []
for sid in series_ids:
    sales = rng.normal(500, 50, 200)
    temperature = rng.normal(20, 5, 200)
    for i, d in enumerate(dates):
        rows.append({"date": d, "store_id": sid, "sales": sales[i],
                     "temperature": temperature[i]})

data_long = pd.DataFrame(rows)

print(f"Shape: {data_long.shape}")
print(f"Unique dates: {data_long['date'].nunique()}")
print(f"Unique stores: {data_long['store_id'].nunique()}")
print()

# Full pipeline — same dates across series should NOT be flagged as duplicates
forecaster_profile = assistant.profile(
    data=data_long,
    target="sales",
    date_column="date",
    series_id_column="store_id",
)

print("Detected profile:")
print(f"  data_format: {forecaster_profile.data_profile.data_format}")
print(f"  frequency: {forecaster_profile.data_profile.frequency}")
print(f"  has_duplicate_timestamps: {forecaster_profile.data_profile.has_duplicate_timestamps}")
print(f"  exog_columns: {forecaster_profile.data_profile.exog_columns}")
print()

plan = assistant.generate_plan(forecaster_profile, steps=14)

print("Preprocessing steps:")
if plan.preprocessing_steps:
    for step in plan.preprocessing_steps:
        print(f"  - [{step.action}] {step.reason}")
else:
    print("  (none — no false duplicate detection)")
print()

code = assistant.generate_code_from_plan(plan=plan, data_profile=forecaster_profile.data_profile)
compile(code, "<demo>", "exec")
print(code)

Shape: (600, 4)
Unique dates: 200
Unique stores: 3

Detected profile:
  data_format: long
  frequency: D
  has_duplicate_timestamps: False
  exog_columns: ['temperature']

Preprocessing steps:
  (none — no false duplicate detection)

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import RollingFeatures, reshape_series_long_to_dict, reshape_exog_long_to_dict
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Load data
data = pd.read_csv('data.csv')
data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')

# Reshape to dict format (optimal for ForecasterRecursiveMultiSeries)
series_dict = reshape_series_long_to_dict(
    data      = data,
    series_id = 'store_id',
    index     = 'date',
    values    = 'sales',
    freq      = 'D',
)

exog_dict = reshape_exog_long_to_dict(
    data      = 

### 6.4 Statistical

In [24]:
import numpy as np
import pandas as pd

# Create a clean single-series dataset (no issues)
data_clean = pd.DataFrame(
    {"y": np.sin(np.linspace(0, 8 * np.pi, 365)) + np.random.default_rng(42).normal(0, 0.3, 365)},
    index=pd.date_range("2023-01-01", periods=365, freq="D"),
)

# Full pipeline
forecaster_profile = assistant.profile(data=data_clean, target="y")

print("Recommended forecaster:", forecaster_profile.forecaster)
print("Recommended estimator:", forecaster_profile.estimator)
print()

plan = assistant.generate_plan(
    forecaster_profile, 
    forecaster='ForecasterStats', 
    steps=30
)

print("Preprocessing steps:", plan.preprocessing_steps)
print()

code = assistant.generate_code_from_plan(plan=plan, data_profile=forecaster_profile.data_profile)
compile(code, "<demo>", "exec")
print(code)

Recommended forecaster: ForecasterRecursive
Recommended estimator: LGBMRegressor

Preprocessing steps: []

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.stats import Arima
from skforecast.recursive import ForecasterStats

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

# Train/test split
end_train = '2023-10-19'  # 80% of data, adjust to change the split point
data_train = data.loc[:end_train]
data_test  = data.loc[data.index > end_train]

print(
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"
)

# Create forecaster (Auto-ARIMA)
forecaster = ForecasterStats(
    estimator = Arima(order=None, seasonal_order=None, m=7),
)

# Fit
forecaster.fit(y=da

### 6.5 Foundation

In [23]:
import numpy as np
import pandas as pd

# Create a clean single-series dataset (no issues)
data_clean = pd.DataFrame(
    {"y": np.sin(np.linspace(0, 8 * np.pi, 365)) + np.random.default_rng(42).normal(0, 0.3, 365)},
    index=pd.date_range("2023-01-01", periods=365, freq="D"),
)

# Full pipeline
forecaster_profile = assistant.profile(data=data_clean, target="y")

print("Recommended forecaster:", forecaster_profile.forecaster)
print("Recommended estimator:", forecaster_profile.estimator)
print()

plan = assistant.generate_plan(
    forecaster_profile, 
    forecaster='ForecasterFoundation', 
    estimator_kwargs={"model_id": "autogluon/chronos-2-base", "context_length": 4096},
    steps=30
)

print("Preprocessing steps:", plan.preprocessing_steps)
print()

code = assistant.generate_code_from_plan(plan=plan, data_profile=forecaster_profile.data_profile)
compile(code, "<demo>", "exec")
print(code)

Recommended forecaster: ForecasterRecursive
Recommended estimator: LGBMRegressor

Preprocessing steps: []

import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.foundation import FoundationModel, ForecasterFoundation

# Load data
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')
data = data.sort_index()

series = data['y']

# Train/test split
end_train = '2023-10-19'  # 80% of data, adjust to change the split point
series_train = series.loc[:end_train]
series_test  = series.loc[series.index > end_train]

# Create foundation model (chronos-2-base)
model = FoundationModel(
    model_id       = 'autogluon/chronos-2-base',
    context_length = 4096,
    device_map     = 'auto',
)

# Create forecaster
forecaster = ForecasterFoundation(estimator=model)

# Fit (stores context only — no training)
forecaster.fit(series=series_train)

# Predict
steps = 3